# GridNet-HD — Fusion heads on frozen backbones (Phase 1: late fusion, concat, K×K window attention)

ต้องมีก่อน: LiDAR cache 27 zones (`GridNet/cache`), checkpoint LiDAR-only (`GridNet/ckpt/litept_s_lidar_v2/best.pth`),
2D cache ของ UPerNet v2 สำหรับ **val 6 zones + train zones ที่จะใช้** (`GridNet/cache2d/upernet_convnext_t_v2/<zone>/`,
จาก colab_train_upernet cell 8) และ **tar ต่อโซน** `<zone>.tar` จาก `colab_compact_cache2d` (รันครั้งเดียวบน CPU runtime) —
ถ้าโซนไหนยังไม่มี tar, cell 5 จะ copy ทีละไฟล์แทน (ช้ากว่ามาก: Drive ให้ ~10–15 MB/s ต่อไฟล์เล็ก แต่ >100 MB/s กับไฟล์ใหญ่ไฟล์เดียว)

ทำอะไร: เทรน head ตัวเล็กบน feature ของ frozen backbones (LitePT-S 72-d + UPerNet prob/feat ผ่าน association 8 views) แบบ zone-major
(โหลด 2D maps ของโซนขึ้น GPU ทีละโซน) → eval บน val 6 zones ทุก pass → `eval/eval.json` + `fused_pred.npy` ต่อโซน
LiDAR-only ของ backbone ถูกวัดบน window ชุดเดียวกันไปด้วย (sanity: ต้อง ≈ 69.9)

| HEAD | input | ~เวลาต่อ head บน A100 (8 train zones, 2 passes, eval 2 ครั้ง) |
|---|---|---|
| late | softmax(LiDAR) ⊕ image probs | ~1.7 ชม. (late_v1: เทรน 0.6 s/it, eval ~20 นาที/ครั้ง) |
| concat | LiDAR feat ⊕ image feat ⊕ probs | ~1.8 ชม. |
| window (K=3/5) | เหมือน concat แต่ attention บน K×K cells | ~2 ชม. |

เตรียมข้อมูล (cell 5) ~45 นาทีต่อ session → **รันหลาย head ใน session เดียว** ด้วย `EXTRA_RUNS` เช่น `window:3:window3_v1`
(รูปแบบ `head:K:run_name[:pose_t:pose_r[:seed]]` คั่นด้วย `,`) · `SHUTDOWN_WHEN_DONE` = ปิด runtime เองเมื่อจบทุก run (ไม่เสีย units ตอนเครื่องว่าง)

**Pose-noise augmentation** (reference remedy): `POSE_NOISE_T = 0.25` (m), `POSE_NOISE_R = 0.25` (deg) → ต้องเปลี่ยน `RUN_NAME`
**Resume**: Run all ซ้ำได้ — แต่ละ run ต่อจาก `last.pth` ของตัวเอง (run ที่จบแล้วจะผ่านไปทันที)

In [ ]:
#@title 1. GPU
import subprocess, torch
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())
assert torch.cuda.get_device_capability()[0] >= 8, "need L4 / A100 (flash-attn)"


In [ ]:
#@title 2. Mount Drive + parameters
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive/GridNet'
CODE_ZIP = f'{DRIVE}/code/gridnet_code.zip'      #@param {type:"string"}
HEAD = 'concat'           #@param ["late", "concat", "window"]
K = 3                     #@param {type:"integer"}
RUN_NAME = 'concat_v1'    #@param {type:"string"}
RUN2D = 'upernet_convnext_t_v2'                  #@param {type:"string"}
BACKBONE_CKPT = f'{DRIVE}/ckpt/litept_s_lidar_v2/best.pth'   #@param {type:"string"}
TRAIN_ZONES = 't1z6a,t2z5,t3z3,t5a1,t5b2,t5c1,t6z2,t6z4'      #@param {type:"string"}
VAL_ZONES = 't1z5b,t1z8,t3z4,t4z1,t5b1,t5b5'                  #@param {type:"string"}
PASSES = 2                #@param {type:"integer"}
ITS_PER_ZONE = 300        #@param {type:"integer"}
HEAD_POINTS = 120000      #@param {type:"integer"}
MAX_POINTS = 600000       #@param {type:"integer"}
EVAL_MAX_POINTS = 900000  #@param {type:"integer"}
LR = 0.001                #@param {type:"number"}
POSE_NOISE_T = 0.0        #@param {type:"number"}
POSE_NOISE_R = 0.0        #@param {type:"number"}
AUG_PROB = 0.5            #@param {type:"number"}
SEED = 0                  #@param {type:"integer"}
MAX_HOURS = 5.5           #@param {type:"number"}
EXTRA_RUNS = ''           #@param {type:"string"}
SHUTDOWN_WHEN_DONE = False #@param {type:"boolean"}
COPY_CACHE2D_LOCAL = True #@param {type:"boolean"}
CACHE_DRIVE = f'{DRIVE}/cache'
CACHE2D_DRIVE = f'{DRIVE}/cache2d/{RUN2D}'
ASSOC_DIR = f'{DRIVE}/fusion/assoc_{RUN2D}'      # <zone>/assoc.npz archive (GPU sessions rebuild the table: faster than reading it)
IMAGEVOTE_DIR = f'{DRIVE}/fusion/imagevote_{RUN2D}'   # val assoc.npz already saved there by colab_fusion
work_of = lambda name: f'{DRIVE}/fusion/{name}'
# the queue: the run of the form fields first, then EXTRA_RUNS = 'head:K:run_name[:pose_t:pose_r[:seed]], ...'
runs = [dict(head=HEAD, k=int(K), name=RUN_NAME, pt=float(POSE_NOISE_T), pr=float(POSE_NOISE_R), seed=int(SEED))]
for spec in [x.strip() for x in EXTRA_RUNS.split(',') if x.strip()]:
    f = [x.strip() for x in spec.split(':')]
    assert len(f) >= 3 and f[0] in ('late', 'concat', 'window'), f"bad EXTRA_RUNS entry {spec!r}: use head:K:run_name[:pose_t:pose_r[:seed]]"
    runs.append(dict(head=f[0], k=int(f[1]), name=f[2], pt=float(f[3]) if len(f) > 3 else 0.0,
                     pr=float(f[4]) if len(f) > 4 else 0.0, seed=int(f[5]) if len(f) > 5 else int(SEED)))
assert len({r['name'] for r in runs}) == len(runs), "every run needs its own run_name"
for d in [ASSOC_DIR] + [work_of(r['name']) for r in runs]: os.makedirs(d, exist_ok=True)
train_zones = [z.strip() for z in TRAIN_ZONES.split(',') if z.strip()]
val_zones = [z.strip() for z in VAL_ZONES.split(',') if z.strip()]
import torch
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
if any(r['head'] == 'window' and r['k'] >= 5 for r in runs) and vram_gb < 30:
    print(f"WARNING: window K>=5 on a {vram_gb:.0f} GB GPU -> set HEAD_POINTS 60000 or use A100")
for i, r in enumerate(runs):
    done = os.path.exists(f"{work_of(r['name'])}/eval/eval.json") and os.path.exists(f"{work_of(r['name'])}/best.pth")
    print(f"run {i + 1}/{len(runs)}: {r['name']}  head {r['head']}" + (f" K={r['k']}" if r['head'] == 'window' else '')
          + (f"  pose noise {r['pt']} m / {r['pr']} deg" if r['pt'] or r['pr'] else '') + f"  seed {r['seed']}"
          + ("   (has results on Drive: resumes / skips)" if done else ''))


In [ ]:
#@title 3. Environment (restore tarball from Drive)
import subprocess, os, time
ENV = '/content/env'; TAR = f'{DRIVE}/env/env_py310_torch241_cu124.tar.gz'
t0 = time.time()
if not os.path.exists(f'{ENV}/bin/python'):
    assert os.path.exists(TAR), "no env tarball on Drive: run colab_train_litept cell 3 once"
    subprocess.run(['tar', '-C', '/content', '-xzf', TAR], check=True); print("env restored from Drive")
PY = f'{ENV}/bin/python'
subprocess.run([PY, '-m', 'pip', 'install', '-q', 'huggingface_hub', 'pillow'], check=True)
print(f"env ready in {(time.time() - t0) / 60:.1f} min")


In [ ]:
#@title 4. Code + LitePT
import subprocess, zipfile, os
os.makedirs('/content/code', exist_ok=True); zipfile.ZipFile(CODE_ZIP).extractall('/content/code')
if not os.path.exists('/content/LitePT'):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/prs-eth/LitePT.git", "/content/LitePT"], check=True)
r = subprocess.run([PY, "-c", "import sys; sys.path.insert(0,'/content/LitePT'); sys.path.insert(0,'/content/code'); "
                    "from litept.model import LitePT; import gridnet.train.engine_fusion; print('imports ok')"], capture_output=True, text=True)
print(r.stdout.strip() or r.stderr[-1500:])


In [ ]:
#@title 5. Data -> local disk: LiDAR caches, 2D caches (zone tars), camera files, backbone checkpoint, existing val associations
import os, sys, shutil, subprocess, time
from concurrent.futures import ThreadPoolExecutor
sys.path.insert(0, '/content/code')
from gridnet.train.maps2d import cached_image_ids, zone_tar_manifest, extract_zone_tar, write_compact
DATA = '/content/data'; os.makedirs(DATA, exist_ok=True)
zones = train_zones + val_zones
t0 = time.time()
for z in zones:
    dst = f'/content/cache/{z}/points.npz'
    if not os.path.exists(dst):
        os.makedirs(f'/content/cache/{z}', exist_ok=True); shutil.copy(f'{CACHE_DRIVE}/{z}/points.npz', dst)
print(f"LiDAR caches ready ({(time.time() - t0) / 60:.1f} min)")
missing = [z for z in zones if zone_tar_manifest(CACHE2D_DRIVE, z) is None and not os.path.isdir(f'{CACHE2D_DRIVE}/{z}')]
assert not missing, f"no 2D cache for {missing} in {CACHE2D_DRIVE}: export them with colab_train_upernet cell 8"
CACHE2D = CACHE2D_DRIVE
if COPY_CACHE2D_LOCAL:
    CACHE2D = '/content/cache2d'; t0 = time.time(); slow = []
    for z in zones:
        if zone_tar_manifest(CACHE2D_DRIVE, z, wait=30) is not None:   # one big sequential read: fast
            n = extract_zone_tar(CACHE2D_DRIVE, z, CACHE2D); how = 'tar'
        else:                                                             # no tar yet: parallel per-file copy (slow)
            ids = cached_image_ids(CACHE2D_DRIVE, z)
            with ThreadPoolExecutor(16) as ex:
                list(ex.map(lambda i: write_compact(CACHE2D_DRIVE, z, i, CACHE2D), ids))
            n = len(cached_image_ids(CACHE2D, z)); how = 'files'; slow.append(z)
        print(f"  {z}: {n} images from {how}  ({(time.time() - t0) / 60:.1f} min)", flush=True)
    print(subprocess.run(['du', '-sh', CACHE2D], capture_output=True, text=True).stdout.strip())
    if slow: print("zones without a tar (slow path) - run colab_compact_cache2d once for:", ','.join(slow))
code = ("import sys; sys.path.insert(0,'/content/code'); from gridnet.train.data2d import fetch_zones; "
        f"fetch_zones({zones!r}, '{DATA}', what=('pose',))")
subprocess.run([PY, '-c', code], check=True); print("camera files ready")
CKPT_LOCAL = '/content/backbone_best.pth'
if not os.path.exists(CKPT_LOCAL): shutil.copy(BACKBONE_CKPT, CKPT_LOCAL)
for z in val_zones:                                   # reuse the clean val associations saved by colab_fusion
    src, dst = f'{IMAGEVOTE_DIR}/{z}/assoc.npz', f'{ASSOC_DIR}/{z}/assoc.npz'
    if os.path.exists(src) and not os.path.exists(dst):
        os.makedirs(f'{ASSOC_DIR}/{z}', exist_ok=True); shutil.copy(src, dst); print(f"  assoc {z}: copied from imagevote")
print("associations present:", [z for z in zones if os.path.exists(f'{ASSOC_DIR}/{z}/assoc.npz')],
      "(missing ones are computed on first use, ~1 min each)")


In [ ]:
#@title 6. Train the queue of heads (each resumes from its own last.pth; evaluates on val after every pass)
import subprocess, time, os
env = dict(os.environ, PYTHONPATH='/content/code', PYTHONWARNINGS='ignore', PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True')
proc, status, t_all = None, {}, time.time()
try:
    for i, r in enumerate(runs):
        print(f"\n===== run {i + 1}/{len(runs)}: {r['name']} (head {r['head']}" + (f" K={r['k']}" if r['head'] == 'window' else '') + ") =====", flush=True)
        cmd = [PY, '/content/code/scripts/train_fusion.py', '--cache', '/content/cache', '--cache2d', CACHE2D, '--data', DATA,
               '--assoc-dir', ASSOC_DIR, '--litept', '/content/LitePT', '--backbone-ckpt', CKPT_LOCAL, '--work-dir', work_of(r['name']),
               '--head', r['head'], '--k', str(r['k']), '--train-zones', ','.join(train_zones), '--val-zones', ','.join(val_zones),
               '--passes', str(PASSES), '--its-per-zone', str(ITS_PER_ZONE), '--head-points', str(HEAD_POINTS),
               '--max-points', str(MAX_POINTS), '--eval-max-points', str(EVAL_MAX_POINTS), '--lr', str(LR),
               '--pose-noise-t', str(r['pt']), '--pose-noise-r', str(r['pr']), '--aug-prob', str(AUG_PROB),
               '--seed', str(r['seed']), '--max-hours', str(MAX_HOURS)]
        t0 = time.time()
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
        for line in proc.stdout:
            if 'Warning' not in line and 'def backward' not in line and '_TORCH_CUSTOM_FWD' not in line: print(line, end='')
        proc.wait(); status[r['name']] = proc.returncode
        print(f"[{r['name']}] exit {proc.returncode} after {(time.time() - t0) / 3600:.2f} h", flush=True)
    print(f"queue finished in {(time.time() - t_all) / 3600:.2f} h: {status}")
except KeyboardInterrupt:
    if proc is not None: proc.terminate(); proc.wait()
    print("stopped by user; last.pth of every run is kept on Drive -> re-run this cell to resume"); raise


In [ ]:
#@title 7. Results of best.pth on the val zones + figure, for every run of the queue (re-evaluates only when needed)
FORCE_EVAL = False        #@param {type:"boolean"}
import subprocess, json, os, sys
from IPython.display import Image, display
env = dict(os.environ, PYTHONPATH='/content/code', PYTHONWARNINGS='ignore', PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True')
table = []
for r in runs:
    work = work_of(r['name']); ev = f'{work}/eval/eval.json'
    if not (os.path.exists(f'{work}/best.pth') or os.path.exists(f'{work}/last.pth')):
        print(f"[{r['name']}] no checkpoint yet"); continue
    have = os.path.exists(ev) and json.load(open(ev)).get('tag') == 'best'     # cell 6 leaves the best checkpoint's evaluation there
    if FORCE_EVAL or not have:
        cmd = [PY, '/content/code/scripts/train_fusion.py', '--cache', '/content/cache', '--cache2d', CACHE2D, '--data', DATA,
               '--assoc-dir', ASSOC_DIR, '--litept', '/content/LitePT', '--backbone-ckpt', CKPT_LOCAL, '--work-dir', work,
               '--head', r['head'], '--k', str(r['k']), '--val-zones', ','.join(val_zones), '--eval-max-points', str(EVAL_MAX_POINTS), '--eval-only']
        x = subprocess.run(cmd, capture_output=True, text=True, env=env)
        print('\n'.join(l for l in x.stdout.splitlines() if 'Warning' not in l)[-3000:], x.stderr[-800:])
    if not os.path.exists(ev): continue
    e = json.load(open(ev))
    print(f"\n[{r['name']}]", {k: e.get(k) for k in ('model', 'tag', 'it')}, f"fused mIoU {e['mIoU']:.2f}  OA {e['OA']:.2f}   LiDAR-only {e['lidar_only']['mIoU']:.2f}")
    print('  fused     ', {k: (None if v is None else round(v, 1)) for k, v in e['per_class'].items()})
    print('  LiDAR-only', {k: (None if v is None else round(v, 1)) for k, v in e['lidar_only']['per_class'].items()})
    print('  per zone (fused, LiDAR-only)', {z: (round(v['mIoU'], 1), round(v['lidar_only_mIoU'], 1)) for z, v in e['per_zone'].items()})
    table.append((r['name'], e['mIoU'], e['lidar_only']['mIoU']))
    fig = f"{work}/eval/{r['name']}_val.png"
    title = f"{r['name']} ({r['head']}" + (f" K={r['k']}" if r['head'] == 'window' else '') + ") — validation zones"
    subprocess.run([sys.executable, '/content/code/scripts/plot_eval.py', '--eval', ev, '--out', fig, '--title', title], env=dict(os.environ, PYTHONPATH='/content/code'))
    display(Image(fig)) if os.path.exists(fig) else None
print('\nsummary:'); [print(f"  {n:16s} fused {m:.2f}   LiDAR-only {l:.2f}   gain {m - l:+.2f}") for n, m, l in table]


In [ ]:
#@title 8. (optional) Shut the runtime down once everything above has finished - saves compute units on unattended runs
if SHUTDOWN_WHEN_DONE:
    from google.colab import drive, runtime
    print("flushing pending writes to Drive ..."); drive.flush_and_unmount()
    print("results are on Drive; shutting the runtime down"); runtime.unassign()
else:
    print("SHUTDOWN_WHEN_DONE is off -> remember to Disconnect and delete runtime")


## หมายเหตุ
- ผลไป Drive `GridNet/fusion/<RUN_NAME>/`: `last.pth`, `best.pth`, `log.csv`, `eval/eval.json`, `eval/<zone>/fused_pred.npy`
- association ต่อโซนมีสำเนาเก็บที่ `GridNet/fusion/assoc_<RUN2D>/<zone>/assoc.npz` แต่ session ที่มี GPU จะคำนวณใหม่ทุกครั้ง (~0.4 นาที/โซน เร็วกว่าอ่านไฟล์ 4–10 GB จาก Drive)
- cell 6 จบด้วยการทำให้ `eval/` เป็นผลของ `best.pth` เสมอ (ไม่ eval ซ้ำถ้ามีอยู่แล้ว) → cell 7 แค่แสดงผล
- `lidar_only` ใน eval.json = frozen backbone บน window ชุดเดียวกัน ต้องได้ ≈ 69.9 ถ้าไม่ใช่ แปลว่าโหลด checkpoint ผิด
- จบแล้ว **Disconnect and delete runtime**